In [1]:
# Setup the Jupyter version of Dash
from dash import Dash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64

# Configure OS routines
import os

# Configure the plotting routines
import pandas as pd

# Import the database access layer for interacting with MongoDB
from crud import AnimalShelter

# Implement the module query layer
from query_service import fetch_filtered_animals, build_aggregation_pipeline

###########################
# Data Manipulation / Model
###########################
   

# Connect to database via CRUD Module
db = AnimalShelter()


# Load all animal records into a DataFrame for the initial dashboard view
df = pd.DataFrame.from_records(db.read({}))

# Remove the MongoDB ObjectId field so the Dash table can render the data safely
df.drop(columns=['_id'], inplace=True, errors='ignore')


#########################
# Dashboard Layout / View
#########################
app = Dash(__name__)

# Load dashboard logo image
image_filename = 'grazioso_logo.png' 
encoded_image = base64.b64encode(open(image_filename, 'rb').read())


app.layout = html.Div([
    html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()), style={'height':'100px'}),
    html.H4(('Dashboard by Dante Nardulli - Capstone'), style={'textAlign': 'center'}),
    html.Center(html.B(html.H1('CS-340/499 Dashboard'))),
    html.Hr(),
    html.Div(
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'All Dogs', 'value': 'all'},
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'disaster'}
            ],
            value='all',
            labelStyle={'display': 'inline-block', 'margin': '10px'}
        )


    ),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
                         editable=False,
                         filter_action="native",
                         sort_action="native",
                         sort_mode="multi",
                         column_selectable="single",
                         row_selectable="single",
                         row_deletable=False,
                         selected_rows=[0],  # Default selection to first row
                         page_action='native',
                         page_current=0,
                         page_size=10,  # Show 10 records per page
                         style_table={'overflowX': 'auto'},
                         style_cell={
                             'minWidth': '150px', 'width': '150px', 'maxWidth': '150px',
                             'whiteSpace': 'normal'
                        },

                    ),

                        
    html.Br(),
    html.Hr(),
# Display the chart and map side by side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])


#############################################
# Interaction Between Components / Controller
#############################################



#Update the data table and reset selected row after filtering.
# Update the data table and reset the selected row after filtering.
@app.callback(
    Output('datatable-id', 'data'),
    Output('datatable-id', 'selected_rows'),
    Input('filter-type', 'value')
)
def update_dashboard(filter_type):
    filtered_df = fetch_filtered_animals(db, filter_type)
    records = filtered_df.to_dict('records')

    if not records:
        return [], []

    return records, [0]

@app.callback(
    Output('graph-id', "children"),
    Input('filter-type', 'value')
)
def update_graphs(filter_type):
    pipeline = build_aggregation_pipeline(filter_type)
    results = db.aggregate(pipeline)

    if not results:
        return html.Div("No data available to generate chart.")

    try:
        dff = pd.DataFrame(results)
        fig = px.pie(
            dff,
            names="_id",
            values="count",
            title="Breed Distribution"
        )
        return dcc.Graph(figure=fig)
    except Exception as e:
        return html.Div(f"Error generating graph: {str(e)}")
    
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if not selected_columns:
        return []
    return [{
        'if': { 'column_id': i },
        'backgroundColor': "#DAD2FF"
    } for i in selected_columns]


# Update the map based on the currently visible table data and selected row
@app.callback(
    Output('map-id', 'children'),
    Input('datatable-id', 'derived_virtual_data'),
    Input('datatable-id', 'derived_virtual_selected_rows')
)
def update_map(view_data, selected_rows):
    if not view_data:
        return html.Div("No location data available.")

    dff = pd.DataFrame(view_data)

    if dff.empty:
        return html.Div("No location data available.")

    required_columns = {'location_lat', 'location_long', 'breed', 'name'}
    if not required_columns.issubset(dff.columns):
        return html.Div("Required location fields are missing.")

    row = selected_rows[0] if selected_rows else 0
    if row >= len(dff):
        row = 0

    selected_record = dff.iloc[row]

    if pd.isna(selected_record['location_lat']) or pd.isna(selected_record['location_long']):
        return html.Div("Selected animal does not have valid coordinates.")

    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[selected_record['location_lat'], selected_record['location_long']],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[selected_record['location_lat'], selected_record['location_long']],
                    children=[
                        dl.Tooltip(str(selected_record['breed'])),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(selected_record['name'] if pd.notna(selected_record['name']) else "Unknown")
                        ])
                    ]
                )
            ]
        )
    ]


app.run(debug=True, host="127.0.0.1", port=8050, jupyter_mode="external")


Dash app running on http://127.0.0.1:8050/
